In [ ]:
import urllib.request

MODEL_PATH = "models/sign_languange_model_efficientnet_b0_trained_experiments.pth"
HAND_MODEL_PATH = "models/hand_landmarker.task"
CAMERA_INDEX = 0

CLASS_NAMES = [
  "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
  "a", "b", "c", "d", "e", "f", "g", "h", "i", "j",
  "k", "l", "m", "n", "o", "p", "q", "r", "s", "t",
  "u", "v", "w", "x", "y", "z",
]

urllib.request.urlretrieve(
  "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
  HAND_MODEL_PATH,
)

('models/hand_landmarker.task', <http.client.HTTPMessage at 0x2ea51764690>)

In [2]:
import cv2
import mediapipe as mp
import torch
from PIL import Image
from torch import nn
from torchvision import models, transforms


In [3]:
def load_model(path, num_classes):
  checkpoint = torch.load(path, map_location="cpu")

  model = models.efficientnet_b0(weights=None)
  model.classifier[1] = nn.Linear(
      model.classifier[1].in_features,
      num_classes
  )

  model.load_state_dict(checkpoint["model_state_dict"])
  model.eval()

  return model

model = load_model(MODEL_PATH, len(CLASS_NAMES))

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

In [5]:
def predict(hand_image):
    image = Image.fromarray(cv2.cvtColor(hand_image, cv2.COLOR_BGR2RGB))
    input_tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]

    best_index = probs.argmax().item()
    return CLASS_NAMES[best_index], probs[best_index].item()

In [6]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
  base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
  running_mode=RunningMode.VIDEO,
  num_hands=1,
)
detector = HandLandmarker.create_from_options(options)

HAND_CONNECTIONS = [
  (0,1),(1,2),(2,3),(3,4),          # ibu jari
  (0,5),(5,6),(6,7),(7,8),          # telunjuk
  (5,9),(9,10),(10,11),(11,12),     # tengah
  (9,13),(13,14),(14,15),(15,16),   # manis
  (13,17),(17,18),(18,19),(19,20),  # kelingking
  (0,17),                           # dasar telapak
]

def draw_landmarks(frame, landmarks):
  h, w, _ = frame.shape
  points = [(int(p.x * w), int(p.y * h)) for p in landmarks]

  for start, end in HAND_CONNECTIONS:
    cv2.line(frame, points[start], points[end], (0, 255, 0), 2)

  for x, y in points:
    cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)

def find_hand_box(landmarks, width, height, padding=20):
  xs = [p.x * width for p in landmarks]
  ys = [p.y * height for p in landmarks]
  x1, x2 = int(min(xs)) - padding, int(max(xs)) + padding
  y1, y2 = int(min(ys)) - padding, int(max(ys)) + padding
  return max(x1, 0), max(y1, 0), x2, y2

In [9]:
import time

cap = cv2.VideoCapture(CAMERA_INDEX)
frame_count = 0
PROCESS_EVERY_N_FRAMES = 10

last_box = None
last_label = ""
last_landmarks = None

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)
    frame_count += 1

    if frame_count % PROCESS_EVERY_N_FRAMES == 0:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        timestamp_ms = int(time.time() * 1000)
        result = detector.detect_for_video(mp_image, timestamp_ms)

        if result.hand_landmarks:
            landmarks = result.hand_landmarks[0]
            last_landmarks = landmarks
            h, w, _ = frame.shape
            x1, y1, x2, y2 = find_hand_box(landmarks, w, h)
            hand_crop = frame[y1:y2, x1:x2]

            if hand_crop.size > 0:
                label, confidence = predict(hand_crop)
                last_box = (x1, y1, x2, y2)
                last_label = f"{label} {confidence:.0%}"
        else:
            last_box = None
            last_landmarks = None

    if last_landmarks:
        draw_landmarks(frame, last_landmarks)

    if last_box:
        x1, y1, x2, y2 = last_box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, last_label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    cv2.imshow("Sign Language", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 